In [ ]:
#Install and import tools
!pip install -q transformers datasets accelerate scikit-learn pandas numpy gradio requests

import os
import time
import pickle
import inspect
import requests
import numpy as np
import pandas as pd
import torch
import gradio as gr

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
# Upload, merge, clean, and save the final dataset

uploaded = files.upload()

files_to_merge = [
    "isizulu_human.csv",
    "isizulu_ai.csv",
    "isixhosa_human.csv",
    "isixhosa_ai.csv",
    "sepedi_human.csv",
    "sepedi_ai.csv"
]

language_map = {
    "isizulu_human.csv": "zul",
    "isizulu_ai.csv": "zul",
    "isixhosa_human.csv": "xho",
    "isixhosa_ai.csv": "xho",
    "sepedi_human.csv": "nso",
    "sepedi_ai.csv": "nso"
}

label_map = {
    "isizulu_human.csv": 0,
    "isizulu_ai.csv": 1,
    "isixhosa_human.csv": 0,
    "isixhosa_ai.csv": 1,
    "sepedi_human.csv": 0,
    "sepedi_ai.csv": 1
}

dataframes = []

for file in files_to_merge:
    if file not in uploaded:
        print(f"Warning: {file} was not uploaded. Trying to read it from Colab storage.")

    df_temp = pd.read_csv(file)
    df_temp.columns = df_temp.columns.str.strip().str.lower()

    if "text" not in df_temp.columns:
        raise ValueError(f"{file} must contain a text column")

    df_temp["language"] = language_map[file]
    df_temp["label"] = label_map[file]

    df_temp = df_temp[["text", "language", "label"]]
    dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)


# Clean the text without deleting useful rows

import re

def clean_text(text):
    text = str(text)

    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("•", " ")

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)

    text = re.sub(r"\b\d+\b", " inani ", text)

    text = re.sub(r"(\w+)-\s+(\w+)", r"\1\2", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["text"] = df["text"].astype(str).str.strip()
df["language"] = df["language"].astype(str).str.lower().str.strip()
df["label"] = df["label"].astype(int)

df["text"] = df["text"].apply(clean_text)

df = df.dropna()
df = df[df["text"] != ""]
df = df.drop_duplicates(subset=["text"])


# Keep up to 1000 human and 1000 AI samples per language

balanced_parts = []

for language in df["language"].unique():
    for label in [0, 1]:
        part = df[(df["language"] == language) & (df["label"] == label)].copy()

        sample_size = min(len(part), 1000)

        part = part.sample(sample_size, random_state=42)
        balanced_parts.append(part)

df = pd.concat(balanced_parts, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df = df[["text", "language", "label"]]

df.to_csv("final_african_ai_detection_dataset.csv", index=False)


# Final checks

print("Merged and cleaned dataset saved as final_african_ai_detection_dataset.csv")

print("\nExamples per language:")
print(df["language"].value_counts())

print("\nExamples per label:")
print(df["label"].value_counts())

print("\nExamples per language and label:")
print(df.groupby(["language", "label"]).size())

print("\nDuplicate texts:")
print(df["text"].duplicated().sum())

print("\nSample human texts:")
print(df[df["label"] == 0]["text"].sample(5, random_state=42).to_string(index=False))

print("\nSample AI texts:")
print(df[df["label"] == 1]["text"].sample(5, random_state=42).to_string(index=False))

Saving sepedi_ai.csv to sepedi_ai.csv
Saving isixhosa_ai.csv to isixhosa_ai.csv
Saving isizulu_ai.csv to isizulu_ai.csv
Saving isixhosa_human.csv to isixhosa_human.csv
Saving isizulu_human.csv to isizulu_human.csv
Saving sepedi_human.csv to sepedi_human.csv
Merged and cleaned dataset saved as final_african_ai_detection_dataset.csv

Examples per language:
language
zul    2000
xho    2000
nso    2000
Name: count, dtype: int64

Examples per label:
label
1    3000
0    3000
Name: count, dtype: int64

Examples per language and label:
language  label
nso       0        1000
          1        1000
xho       0        1000
          1        1000
zul       0        1000
          1        1000
dtype: int64

Duplicate texts:
0

Sample human texts:
Uma amagumbi ezihambeli engaphezu kwalokhu, ibh...
   Se se a tshephiša mo ngwa-geng wo re o lebileng
Umshushisiomkhulu womphakathi noma umphathi wen...
"Ngokwami, namuhla kube wusuku oluhle kakhu lu ...
" Xa sele bebhalisile, i-DBE iza kunceda umfund

In [ ]:
#Load and check the merged dataset
df = pd.read_csv("final_african_ai_detection_dataset.csv")

df.columns = df.columns.str.strip().str.lower()

required_columns = {"text", "language", "label"}

if not required_columns.issubset(df.columns):
    raise ValueError("Your CSV must contain these columns: text, language, label")

df = df[["text", "language", "label"]].dropna()

df["text"] = df["text"].astype(str).str.strip()
df["language"] = df["language"].astype(str).str.lower().str.strip()
df["label"] = df["label"].astype(int)

print("Dataset loaded successfully.")
print(df.head())

print("\nExamples per language:")
print(df["language"].value_counts())

print("\nExamples per label:")
print(df["label"].value_counts())

print("\nExamples per language and label:")
print(df.groupby(["language", "label"]).size())

import re

def clean_for_fairness(text):
    text = str(text)
    # Remove digits — present in human, absent in AI
    text = re.sub(r'\d+', '', text)
    # Remove bullet points
    text = re.sub(r'•|\*|-\s', '', text)
    # Remove quotes
    text = re.sub(r'[""\'\"'']', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["text"] = df["text"].apply(clean_for_fairness)

# Also trim long texts to cap the word count gap
def trim_to_max_words(text, max_words=60):
    words = text.split()
    return ' '.join(words[:max_words])

df["text"] = df["text"].apply(trim_to_max_words)

Dataset loaded successfully.
                                                text language  label
0  Umeluleki ngokwengqondo uBoitumelo Tlhapane, o...      zul      1
1  Ngelo xesha ke waye wazibophelela ngelithi uku...      xho      1
2  Uchaze nokuthi u mnyango uthatha indlela ephel...      zul      0
3  Lungisa indlu yangasese evuzayo kungenjalo ino...      xho      0
4  Go beakanya go ba benggae, dihlopha di tsenetš...      nso      1

Examples per language:
language
zul    2000
xho    2000
nso    2000
Name: count, dtype: int64

Examples per label:
label
1    3000
0    3000
Name: count, dtype: int64

Examples per language and label:
language  label
nso       0        1000
          1        1000
xho       0        1000
          1        1000
zul       0        1000
          1        1000
dtype: int64


In [ ]:
#Split the data by language
languages = sorted(df["language"].unique())
splits = {}

for language in languages:
    language_data = df[df["language"] == language].reset_index(drop=True)

    if language_data["label"].nunique() < 2:
        print(f"Skipping {language} because it does not have both human and AI examples.")
        continue

    train_data, temp_data = train_test_split(
        language_data,
        test_size=0.2,
        stratify=language_data["label"],
        random_state=42
    )

    validation_data, test_data = train_test_split(
        temp_data,
        test_size=0.5,
        stratify=temp_data["label"],
        random_state=42
    )

    splits[language] = {
        "train": train_data.reset_index(drop=True),
        "validation": validation_data.reset_index(drop=True),
        "test": test_data.reset_index(drop=True)
    }

    print(
        f"{language}: train={len(train_data)}, "
        f"validation={len(validation_data)}, test={len(test_data)}"
    )

nso: train=1600, validation=200, test=200
xho: train=1600, validation=200, test=200
zul: train=1600, validation=200, test=200


In [ ]:
#Set up evaluation
all_results = []

def evaluate_model(y_true, y_pred, model_name, language):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\nModel: {model_name}")
    print(f"Language: {language}")
    print(f"Macro-F1:  {macro_f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(classification_report(y_true, y_pred, target_names=["Human", "AI"]))

    return {
        "model": model_name,
        "language": language,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# Train Logistic Regression baseline models

os.makedirs("models/logistic_regression", exist_ok=True)

lr_models = {}

for language, data in splits.items():
    print(f"\nTraining Logistic Regression for {language}...")

    vectorizer = TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 1),
        lowercase=True
    )

    X_train = vectorizer.fit_transform(data["train"]["text"])
    X_test = vectorizer.transform(data["test"]["text"])

    y_train = data["train"]["label"]
    y_test = data["test"]["label"]

    lr_model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.3
    )

    lr_model.fit(X_train, y_train)

    predictions = lr_model.predict(X_test)

    result = evaluate_model(
        y_true=y_test,
        y_pred=predictions,
        model_name="Logistic Regression",
        language=language
    )

    all_results.append(result)

    lr_models[language] = {
        "vectorizer": vectorizer,
        "model": lr_model
    }

    with open(f"models/logistic_regression/{language}_lr_model.pkl", "wb") as file:
        pickle.dump(lr_models[language], file)


Training Logistic Regression for nso...

Model: Logistic Regression
Language: nso
Macro-F1:  0.7600
Accuracy:  0.7600
Precision: 0.7601
Recall:    0.7600
              precision    recall  f1-score   support

       Human       0.77      0.75      0.76       100
          AI       0.75      0.77      0.76       100

    accuracy                           0.76       200
   macro avg       0.76      0.76      0.76       200
weighted avg       0.76      0.76      0.76       200


Training Logistic Regression for xho...

Model: Logistic Regression
Language: xho
Macro-F1:  0.6476
Accuracy:  0.6550
Precision: 0.6692
Recall:    0.6550
              precision    recall  f1-score   support

       Human       0.62      0.80      0.70       100
          AI       0.72      0.51      0.60       100

    accuracy                           0.66       200
   macro avg       0.67      0.66      0.65       200
weighted avg       0.67      0.66      0.65       200


Training Logistic Regression for zu

In [ ]:
#Prepare AfroXLMR
afroxlmr_name = "Davlan/afro-xlmr-base"

tokenizer = AutoTokenizer.from_pretrained(afroxlmr_name)

def tokenize_text(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

def calculate_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "accuracy": accuracy_score(labels, predictions)
    }

def create_training_settings(output_folder):
    training_args_parameters = inspect.signature(TrainingArguments.__init__).parameters

    settings = {
        "output_dir": output_folder,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "logging_steps": 20,
        "report_to": "none",
        "fp16": torch.cuda.is_available()
    }

    if "evaluation_strategy" in training_args_parameters:
        settings["evaluation_strategy"] = "epoch"
    else:
        settings["eval_strategy"] = "epoch"

    return TrainingArguments(**settings)

def prepare_dataset(dataframe):
    dataset = Dataset.from_pandas(dataframe[["text", "label"]])
    dataset = dataset.map(tokenize_text, batched=True)
    dataset = dataset.remove_columns(["text"])
    dataset.set_format("torch")
    return dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Train AfroXLMR per language

os.makedirs("models/afroxlmr_monolingual", exist_ok=True)

for language, data in splits.items():
    print(f"\nFine-tuning AfroXLMR for {language}...")

    train_dataset = prepare_dataset(data["train"])
    validation_dataset = prepare_dataset(data["validation"])
    test_dataset = prepare_dataset(data["test"])

    model = AutoModelForSequenceClassification.from_pretrained(
        afroxlmr_name,
        num_labels=2
    )

    training_settings = create_training_settings(
        f"models/afroxlmr_monolingual/{language}"
    )

    trainer_kwargs = {
        "model": model,
        "args": training_settings,
        "train_dataset": train_dataset,
        "eval_dataset": validation_dataset,
        "compute_metrics": calculate_metrics
    }

    trainer_parameters = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_parameters:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train()

    test_output = trainer.predict(test_dataset)

    predictions = np.argmax(test_output.predictions, axis=-1)
    true_labels = test_output.label_ids

    result = evaluate_model(
        y_true=true_labels,
        y_pred=predictions,
        model_name="AfroXLMR Monolingual",
        language=language
    )

    all_results.append(result)

    trainer.save_model(f"models/afroxlmr_monolingual/{language}/best_model")


Fine-tuning AfroXLMR for nso...


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Model: AfroXLMR Monolingual
Language: zul
Macro-F1:  0.9950
Accuracy:  0.9950
Precision: 0.9950
Recall:    0.9950
              precision    recall  f1-score   support

       Human       0.99      1.00      1.00       100
          AI       1.00      0.99      0.99       100

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Train multilingual AfroXLMR

os.makedirs("models/afroxlmr_multilingual", exist_ok=True)

combined_train_data = pd.concat(
    [splits[language]["train"] for language in splits],
    ignore_index=True
)

combined_validation_data = pd.concat(
    [splits[language]["validation"] for language in splits],
    ignore_index=True
)

combined_train_data = combined_train_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nTraining multilingual AfroXLMR model...")
print("Training examples:", len(combined_train_data))
print("Validation examples:", len(combined_validation_data))

multilingual_train_dataset = prepare_dataset(combined_train_data)
multilingual_validation_dataset = prepare_dataset(combined_validation_data)

multilingual_model = AutoModelForSequenceClassification.from_pretrained(
    afroxlmr_name,
    num_labels=2
)

multilingual_training_settings = create_training_settings(
    "models/afroxlmr_multilingual"
)

trainer_kwargs = {
    "model": multilingual_model,
    "args": multilingual_training_settings,
    "train_dataset": multilingual_train_dataset,
    "eval_dataset": multilingual_validation_dataset,
    "compute_metrics": calculate_metrics
}

trainer_parameters = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer

multilingual_trainer = Trainer(**trainer_kwargs)

multilingual_trainer.train()

multilingual_trainer.save_model("models/afroxlmr_multilingual/best_model")
tokenizer.save_pretrained("models/afroxlmr_multilingual/best_model")


Training multilingual AfroXLMR model...
Training examples: 4800
Validation examples: 600


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/afroxlmr_multilingual/best_model/tokenizer_config.json',
 'models/afroxlmr_multilingual/best_model/tokenizer.json')

In [ ]:
#Test multilingual model per language
for language, data in splits.items():
    print(f"\nTesting multilingual AfroXLMR on {language}...")

    test_dataset = prepare_dataset(data["test"])
    test_output = multilingual_trainer.predict(test_dataset)

    predictions = np.argmax(test_output.predictions, axis=-1)
    true_labels = test_output.label_ids

    result = evaluate_model(
        y_true=true_labels,
        y_pred=predictions,
        model_name="AfroXLMR Multilingual",
        language=language
    )

    all_results.append(result)


Testing multilingual AfroXLMR on nso...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Multilingual
Language: nso
Macro-F1:  0.9950
Accuracy:  0.9950
Precision: 0.9950
Recall:    0.9950
              precision    recall  f1-score   support

       Human       0.99      1.00      1.00       100
          AI       1.00      0.99      0.99       100

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200


Testing multilingual AfroXLMR on xho...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Multilingual
Language: xho
Macro-F1:  1.0000
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       100
          AI       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


Testing multilingual AfroXLMR on zul...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Multilingual
Language: zul
Macro-F1:  1.0000
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       100
          AI       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [ ]:
# Run cross-lingual testing

print("\nStarting cross-lingual testing...")

for train_language in splits:
    trained_model_path = f"models/afroxlmr_monolingual/{train_language}/best_model"

    transfer_model = AutoModelForSequenceClassification.from_pretrained(
        trained_model_path
    )

    trainer_kwargs = {
        "model": transfer_model,
        "args": TrainingArguments(
            output_dir=f"models/cross_lingual_{train_language}",
            report_to="none"
        ),
        "compute_metrics": calculate_metrics
    }

    trainer_parameters = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_parameters:
        trainer_kwargs["tokenizer"] = tokenizer

    transfer_trainer = Trainer(**trainer_kwargs)

    for test_language, data in splits.items():
        if train_language == test_language:
            continue

        print(f"\nTesting {train_language} model on {test_language} data...")

        test_dataset = prepare_dataset(data["test"])
        test_output = transfer_trainer.predict(test_dataset)

        predictions = np.argmax(test_output.predictions, axis=-1)
        true_labels = test_output.label_ids

        result = evaluate_model(
            y_true=true_labels,
            y_pred=predictions,
            model_name=f"AfroXLMR Cross-lingual: {train_language} to {test_language}",
            language=test_language
        )

        all_results.append(result)


Starting cross-lingual testing...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Testing nso model on xho data...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Cross-lingual: nso to xho
Language: xho
Macro-F1:  0.8887
Accuracy:  0.8900
Precision: 0.9098
Recall:    0.8900
              precision    recall  f1-score   support

       Human       1.00      0.78      0.88       100
          AI       0.82      1.00      0.90       100

    accuracy                           0.89       200
   macro avg       0.91      0.89      0.89       200
weighted avg       0.91      0.89      0.89       200


Testing nso model on zul data...


...


Testing zul model on nso data...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Cross-lingual: zul to nso
Language: nso
Macro-F1:  0.9950
Accuracy:  0.9950
Precision: 0.9950
Recall:    0.9950
              precision    recall  f1-score   support

       Human       0.99      1.00      1.00       100
          AI       1.00      0.99      0.99       100

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200


Testing zul model on xho data...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Model: AfroXLMR Cross-lingual: zul to xho
Language: xho
Macro-F1:  1.0000
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       100
          AI       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [ ]:
RUN_GPTZERO = False

GPTZERO_API_KEY = "PASTE_YOUR_API_KEY_HERE"
GPTZERO_URL = "https://api.gptzero.me/v2/predict/text"

def predict_with_gptzero(text):
    try:
        response = requests.post(
            GPTZERO_URL,
            headers={
                "x-api-key": GPTZERO_API_KEY,
                "Content-Type": "application/json"
            },
            json={"document": text},
            timeout=20
        )

        result = response.json()
        ai_probability = result["documents"][0]["completely_generated_prob"]

        return 1 if ai_probability >= 0.5 else 0

    except Exception as error:
        print("GPTZero error:", error)
        return None


if RUN_GPTZERO:
    for language, data in splits.items():
        print(f"\nTesting GPTZero on {language}...")

        sample_data = data["test"].sample(
            min(50, len(data["test"])),
            random_state=42
        )

        true_labels = []
        predictions = []

        for _, row in sample_data.iterrows():
            prediction = predict_with_gptzero(row["text"])

            if prediction is not None:
                true_labels.append(row["label"])
                predictions.append(prediction)

            time.sleep(0.5)

        if len(predictions) > 0:
            result = evaluate_model(
                y_true=true_labels,
                y_pred=predictions,
                model_name="GPTZero",
                language=language
            )

            all_results.append(result)

else:
    print("\nGPTZero skipped.")

In [ ]:
results_df = pd.DataFrame(all_results)

print("\nFinal results:")
display(results_df)

results_df.to_csv("cos760_group22_results.csv", index=False)

files.download("cos760_group22_results.csv")


Final results:


,model,language,macro_f1,accuracy,precision,recall
0,Logistic Regression,nso,0.759976,0.760,0.760104,0.760
1,Logistic Regression,xho,0.647591,0.655,0.669232,0.655
2,Logistic Regression,zul,0.625000,0.640,0.666667,0.640
3,AfroXLMR Monolingual,nso,0.995000,0.995,0.995050,0.995
4,AfroXLMR Monolingual,xho,0.995000,0.995,0.995050,0.995
5,AfroXLMR Monolingual,zul,0.995000,0.995,0.995050,0.995
6,AfroXLMR Multilingual,nso,0.995000,0.995,0.995050,0.995
7,AfroXLMR Multilingual,xho,1.000000,1.000,1.000000,1.000
8,AfroXLMR Multilingual,zul,1.000000,1.000,1.000000,1.000
9,AfroXLMR Cross-lingual: nso to xho,xho,0.888653,0.890,0.909836,0.890


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
final_model_path = "models/afroxlmr_multilingual/best_model"

final_tokenizer = AutoTokenizer.from_pretrained(final_model_path)
final_model = AutoModelForSequenceClassification.from_pretrained(final_model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_model.to(device)
final_model.eval()

def predict_text_type(text):
    if text is None or text.strip() == "":
        return {
            "Human-written": 0.0,
            "AI-generated": 0.0
        }

    inputs = final_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = final_model(**inputs)
        probabilities = torch.softmax(output.logits, dim=-1).squeeze().cpu().tolist()

    return {
        "Human-written": round(probabilities[0], 6),
        "AI-generated": round(probabilities[1], 6)
    }


demo = gr.Interface(
    fn=predict_text_type,
    inputs=gr.Textbox(
        lines=7,
        label="Text",
        placeholder="Paste isiZulu, isiXhosa, or Sepedi text here..."
    ),
    outputs=gr.Label(label="Confidence scores"),
    title="African Language AI Text Detector",
    description=(
        "This demo shows the confidence that a text is human-written or AI-generated. "
        "It uses the final multilingual AfroXLMR model trained on isiZulu, isiXhosa and Sepedi."
    ),
    submit_btn="Detect",
    clear_btn="Clear"
)

demo.launch(share=True, debug=True, show_error=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://869bcac7afae3a02c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
